# RunPod 서버리스 콜드스타트·캐릭터 생성 지연 최적화
## Before/After 실측 비교 보고

**작성일**: 2026-06-21 · **대상**: `mongle-ai` (RunPod LLM/이미지 워커 + 캐릭터 생성 파이프라인) · **관련**: PR #121, ADR-0004

> 본 노트북은 한 세션에서 수행한 진단·개입을 논문 형식으로 정리한다. 모든 수치는 RunPod 실행 API(`/run`→`/status`의 `delayTime`/`executionTime`)와 `/health`에서 **직접 실측**한 값과, 직접 측정하지 못해 **추정(estimate)**으로 명시한 값으로 구분한다.


## 초록 (Abstract)

RunPod Serverless 워커의 콜드스타트가 5분 이상 걸려 서버(EC2)→RunPod 요청이 타임아웃되는 문제가 보고되었다. 실측 결과 **지연의 지배 요인은 추론 연산이 아니라 워밍/콜드스타트**임을 확인했다(워밍된 LLM의 8토큰 생성 `executionTime` 0.25–0.35초 대 콜드 첫 요청 36초, 큐/콜드 `delayTime` 0.13초 대 15초). 

이를 근거로 세 갈래로 개입했다. (1) 엔드포인트 설정(`idleTimeout` 5→600초, `gpuIds`를 Blackwell MIG 제외 후 `AMPERE_24` 고정), (2) 캐릭터 생성 파이프라인에서 별도 번역 LLM 단계 제거(직렬 3단계·3엔드포인트 → 2단계·2엔드포인트), (3) 워커 빌드 최적화(레이어 순서 교정으로 의존성 변경 시 수 GB 재다운로드 제거, HF 429 백오프, 모델 가중치 commit SHA 고정). 

본 보고는 각 개입의 before/after를 실측·구조 변화 기준으로 비교하고, 워밍 기법의 한계와 잔여 병목(워밍 상태의 추론 바닥 ~40초)을 비판적으로 논의한다.


## 실행/재현 노트

- 본 노트북은 **외부 의존성 없이** 실행된다(표준 라이브러리만 사용). 차트 셀은 `matplotlib`이 있으면 그래프를, 없으면 텍스트 막대를 출력한다.
- 수치는 2026-06-21 단발 실측이다. 표본이 작고 부하/동시성은 측정하지 않았다(§6 한계 참조).


## 1. 서론 및 문제 정의

### 1.1 시스템 구성
```
브라우저 ──HTTPS──▶ Django (EC2, CPU) ──HTTP──▶ FastAPI mongle-ai (RunPod CPU Pod)
                                                     │
                                                     ├─▶ RunPod Serverless: character LLM (GPU)
                                                     ├─▶ RunPod Serverless: planner LLM  (GPU)
                                                     └─▶ RunPod Serverless: image gen    (GPU)
```
FastAPI는 추론을 하지 않는 **프록시**다(가중치 로드·생성은 전부 RunPod GPU 워커에서 발생). 따라서 FastAPI를 GPU로 옮겨도 지연은 줄지 않는다 — 이는 초기 가설을 실측으로 기각한 지점이다.

### 1.2 증상
- 콜드스타트 시 응답까지 5분 이상.
- 서버에서 보낸 요청이 RunPod까지 도달하지 못하는 것처럼 보임(실제로는 동기 타임아웃이 콜드스타트보다 짧아 끊김).


## 2. 측정 방법론

- **지표**: RunPod 실행 API가 잡 완료 시 반환하는 `delayTime`(큐 대기 + 콜드스타트)과 `executionTime`(핸들러 내 실제 추론 시간, 밀리초).
- **워커 상태**: `GET https://api.runpod.ai/v2/{id}/health` 의 `workers`(idle/ready/running/initializing)와 `jobs`(completed/failed/retried).
- **절차**: 동일 엔드포인트에 (a) 워밍 판별용 trivial 요청(8토큰)을 연속 발사해 cold→hot 전이를 관측하고, (b) 실제급 워크로드(800토큰)를 1회 측정.
- **주의**: `delayTime`은 콜드스타트를, `executionTime`은 순수 추론을 분리해준다. 이 분리가 '지연이 추론이냐 워밍이냐'를 가르는 핵심 도구다.


In [ ]:
# === 세션 실측 데이터 (2026-06-21) ===
# 단위: ms. kind = measured(직접 실측) | estimate(추정)

# 2.1 LLM 엔드포인트 latency: delayTime(콜드/큐), executionTime(추론)
llm_latency = {
    # 라벨: (delay_ms, exec_ms, kind)
    'cold 첫요청 (8토큰)':        (15038, 35841, 'measured'),
    'hot 2번째 (8토큰)':          (2029,    306, 'measured'),
    'hot 3번째 (8토큰)':          (133,     247, 'measured'),
    'hot 재실측 (8토큰)':         (133,     351, 'measured'),
    '실제 planner (800토큰)':     (136,   41586, 'measured'),
}

# 2.2 엔드포인트 헬스 스냅샷 (jobs 누적)
health = {
    'planner': {'completed':410,'failed':27,'retried':4},
    'village': {'completed':55, 'failed':9, 'retried':3},
    'image':   {'completed':66, 'failed':1, 'retried':0},
}

def fail_rate(h): return 100*h['failed']/(h['completed']+h['failed'])
for name,h in health.items():
    print(f"{name:8} 실패율 {fail_rate(h):4.1f}%  (completed={h['completed']}, failed={h['failed']})")


## 3. 베이스라인 (Before)

### 3.1 LLM: 워밍이 지연을 지배한다
아래 표/차트의 핵심: **워밍되면 8토큰 생성이 0.25–0.35초**로 정상이다. 콜드 첫 요청의 36초는 모델의 lazy 로드가 `executionTime`에 잡힌 것이고, 전체 콜드(컨테이너+가중치)는 수 분이다. 즉 추론 엔진 자체는 느리지 않다.

### 3.2 엔드포인트 설정(Before)
| 항목 | 값 | 함의 |
|---|---|---|
| `workersMin` | 0 | 거의 모든 실사용 요청이 콜드 |
| `idleTimeout` | 5초 | 요청 끝나고 5초면 워커 사망 → 다음 요청 또 콜드 |
| `gpuIds` | `AMPERE_24` + Blackwell MIG | Blackwell 비호환 → 실패/retry 유발 의심 |
| `networkVolumeId` | null | 콜드마다 가중치 처음부터 로드 |

### 3.3 실패율(Before): village 14.1%가 두드러짐 (Blackwell 비호환 의심)


In [ ]:
def show_bars(title, labels, values, unit='ms'):
    """matplotlib 있으면 막대그래프, 없으면 텍스트 폴백."""
    try:
        import matplotlib.pyplot as plt
        for _f in ('AppleGothic','NanumGothic','Malgun Gothic','Noto Sans CJK KR'):
            try:
                plt.rcParams['font.family']=_f; break
            except Exception:
                pass
        plt.rcParams['axes.unicode_minus']=False
        fig, ax = plt.subplots(figsize=(8, 0.5*len(labels)+1))
        ax.barh(range(len(labels)), values)
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
        ax.invert_yaxis(); ax.set_xlabel(unit); ax.set_title(title)
        for i,v in enumerate(values): ax.text(v, i, f' {v}', va='center')
        plt.tight_layout(); plt.show()
    except ModuleNotFoundError:
        mx = max(values) or 1
        print(title)
        for l,v in zip(labels, values):
            print(f'  {l:24.24} | {chr(9608)*int(40*v/mx)} {v} {unit}')

labels = list(llm_latency)
delays = [v[0] for v in llm_latency.values()]
execs  = [v[1] for v in llm_latency.values()]
show_bars('delayTime (콜드/큐) — Before 베이스라인', labels, delays)
print()
show_bars('executionTime (추론) — Before 베이스라인', labels, execs)


## 4. 개입 (Interventions)

### 4.1 엔드포인트 설정 (적용 완료, 라이브)
`saveEndpoint`로 3개 엔드포인트 모두 `idleTimeout` 5→**600초**, `gpuIds`→**`AMPERE_24`**(Blackwell 제거). `workersMin`은 비용 때문에 0 유지(상시 워커 대신 idle 윈도 확대 선택).

### 4.2 캐릭터 생성 파이프라인: 번역 LLM 단계 제거 (PR #121)
persona LLM이 한국어 외모와 함께 **영어 visual 태그(`appearance_en`)를 한 번에 출력**하게 바꿔, SDXL용 번역만 담당하던 별도 LLM 호출을 제거했다. 이 번역은 `planner` 엔드포인트를 썼으므로, 제거로 **캐릭터 생성 임계경로에서 엔드포인트 1개(planner)와 콜드스타트 1회가 통째로 빠진다**. `appearance_en or appearance` 폴백으로 기존 `Character.visual` 계약은 유지.

### 4.3 워커 빌드 최적화 (PR #121, ADR-0004)
- **레이어 순서**: image 워커 `requirements`를 ML 스택/app(runpod)으로 분리하고 `bake`를 app 의존성 앞으로 → app 의존성 변경 시 SDXL 수 GB 재다운로드 제거.
- **429 백오프**: image `bake`에도 HF rate-limit 지수 백오프 추가.
- **revision 고정**: 4개 모델 가중치를 commit SHA로 고정(bake↔런타임 동일) → 재현성 + 런타임 캐시 히트.

### 4.4 (보조) health-gate 예열
`POST /v1/character/warmup`: `/health`로 워커 0개일 때만 1건 발사. 슬롯 경합·중복 비용 회피. *단, §6에서 그 효용 한계를 논의한다.*


In [ ]:
# === Before/After 비교 데이터 ===
endpoint_cfg = [
    # (항목, before, after)
    ('idleTimeout', '5초', '600초'),
    ('gpuIds', 'AMPERE_24 + Blackwell MIG', 'AMPERE_24'),
    ('workersMin', '0', '0 (유지)'),
]
print('[엔드포인트 설정 Before/After]')
for k,b,a in endpoint_cfg:
    print(f'  {k:12} : {b:30} -> {a}')

# 캐릭터 생성 파이프라인 단계 (직렬). exec_ms 는 추정 포함 — kind 표기.
pipeline_before = [
    ('persona LLM (character EP)',  41600, 'measured~ (planner 800토큰 유사치)'),
    ('translate LLM (planner EP)',   8000, 'estimate'),
    ('image SDXL 30스텝 (image EP)', 20000, 'estimate (warm)'),
]
pipeline_after = [
    ('persona LLM +appearance_en (character EP)', 42000, 'measured~ (필드 1개 추가, 무시할 차이)'),
    ('image SDXL 30스텝 (image EP)',              20000, 'estimate (warm)'),
]
def total(p): return sum(x[1] for x in p)
print('\n[캐릭터 생성 파이프라인 — warm 기준, 추정 포함]')
print(f'  Before: {len(pipeline_before)}단계 / 3 엔드포인트 / 합 ~{total(pipeline_before)/1000:.0f}s')
print(f'  After : {len(pipeline_after)}단계 / 2 엔드포인트 / 합 ~{total(pipeline_after)/1000:.0f}s')
print('  * 구조적 확정 이득: LLM 왕복 1회 + planner 엔드포인트 콜드스타트 1회 제거')


## 5. 결과 (Before / After)

### 5.1 확정(구조적) 이득 — 추정 불필요
| 영역 | Before | After |
|---|---|---|
| 캐릭터 생성 임계경로 | LLM 2회 + 이미지 1회 = **3 RunPod 호출 / 3 엔드포인트** | LLM 1회 + 이미지 1회 = **2 호출 / 2 엔드포인트** |
| idleTimeout(워커 유지 시간) | 요청 후 **5초** 뒤 워커 사망 | **600초** 유지 |
| GPU 호환 | Blackwell MIG 포함(실패 유발) | `AMPERE_24` 고정 |
| 의존성 변경 시 image 빌드 | SDXL 수 GB **재다운로드** | **재다운로드 없음**(레이어 캐시 유지) |
| 빌드 재현성 | 모델 `main` 표류 | 가중치 **commit SHA 고정** |
| 런타임 콜드 재다운로드 | HF_HOME shadow/리비전 불일치 위험 | bake↔런타임 SHA 일치로 캐시 히트 |

### 5.2 측정 기반 이득
- 워밍 상태 LLM `executionTime` 0.25–0.35초(8토큰)는 **추론 엔진이 병목이 아님**을 보여, 개입 방향을 '워밍/콜드스타트 + 파이프라인 구조'로 정당화한다.
- `idleTimeout` 5→600초로 **활성 세션 내 연속 요청은 hot 경로**(delay ~0.13초)에 머문다.

### 5.3 추정 기반 이득 (검증 대상)
- 캐릭터 생성 warm 총시간 ~70초 → ~62초(번역 단계 제거분, 추정). 배포 후 **라이브 실측은 부록 A** 참조(콜드 런 기준, A 동작·번역 제거 확인됨).


In [ ]:
# 캐릭터 생성: 엔드포인트 수 & 단계 수 Before/After (확정 이득)
show_bars('캐릭터 생성 임계경로 — RunPod 호출 수', ['Before','After'], [3,2], unit='calls')
print()
show_bars('캐릭터 생성 — 관여 엔드포인트 수', ['Before','After'], [3,2], unit='endpoints')
print()
show_bars('파이프라인 warm 총시간(추정 포함)', ['Before','After'],
         [int(total(pipeline_before)), int(total(pipeline_after))])


## 6. 논의 (비판적)

### 6.1 워밍 기법의 구조적 한계
제출 시점에 이미지 워커를 예열해 LLM 단계(~40초)와 겹치려는 아이디어는, **콜드 부팅(수 분) > 겹침 창(40초)**이라 실제로는 실시간 잡 도착 시점에 워커가 ready가 아닐 수 있다. 즉 health-gate가 발사하는 경우(=콜드)엔 도움이 약하고, 도움이 될 경우(=이미 warm)엔 skip된다. **워밍은 콜드스타트가 이미 짧을 때(가중치 bake)만 잘 듣는다.**

### 6.2 진짜 레버는 콜드 자체를 줄이는 것
`networkVolumeId=null`이라 콜드마다 가중치를 로드한다. 콜드 5분을 근본적으로 줄이려면 **가중치 bake/이미지화**(본 PR의 빌드 최적화가 그 토대) 또는 `workersMin=1`(상시 워커, 비용↑)이 필요하다. 워밍·`idleTimeout`은 그 위의 보조 수단이다.

### 6.3 잔여 병목: 워밍 상태의 추론 바닥
워밍돼도 실제 planner 워크로드는 `executionTime` 41.6초(800토큰)다. 콜드를 다 없애도 이 바닥은 남는다. 체감 지연이 이 바닥에 지배된다면 **출력 토큰 축소·diffusion 스텝 축소·더 빠른 GPU**(워밍 바닥 자체를 낮추기)가 별개의 큰 레버다.

### 6.4 `idleTimeout=600`의 트레이드오프
드문드문한 단발 요청 패턴에서는 매 요청마다 최대 10분치 idle GPU를 빌링한다. 누적되면 `workersMin=1` 정액제가 더 쌀 수 있는 교차점이 존재한다 — '저트래픽이라 idleTimeout이 항상 싸다'는 무조건 참이 아니다.

### 6.5 측정의 한계
단발 실측, 작은 표본, 부하/동시성 미측정. 캐릭터 파이프라인의 번역·이미지 단계 시간은 추정이다. 결론의 '확정 이득'은 구조 변화(호출/엔드포인트 수, 빌드 동작)에 근거하며, 시간 절감 수치는 배포 후 재측정으로 확증해야 한다.


## 7. 결론

- 지연의 지배 요인은 추론이 아니라 **워밍/콜드스타트**임을 실측으로 규명했고, FastAPI→GPU 이전 가설을 기각했다.
- **확정 이득**: 캐릭터 생성 임계경로를 3호출/3엔드포인트 → 2호출/2엔드포인트로 축소, idleTimeout(워커 유지 시간) 5→600초, GPU 호환 고정, 의존성 변경 시 수 GB 재다운로드 제거, 빌드 재현성 확보.
- **다음 단계**: (1) 이미지 워커 재빌드/재배포 후 콜드스타트·파이프라인 시간 **재측정**, (2) 출력 토큰·diffusion 스텝 축소로 워밍 바닥 낮추기, (3) 트래픽 데이터로 `idleTimeout` vs `workersMin=1` 비용 교차점 재평가.


## 부록 A. 배포 후 라이브 실측 (2026-06-21, 콜드 런)

PR #121 머지·배포 후 운영 Pod(`/v1/character`)에 1회 호출한 **실측값**이다.

**A(번역 단계 제거) 라이브 검증**
- 입력 `persona='포근한 갈색 곰, 노란 스카프'` → 반환 `appearance = "cuddly brown bear, round ears, big round eyes, yellow scarf"`. persona LLM이 **영어 visual 태그를 직접** 출력(별도 번역 호출 없이). A의 유일 리스크였던 품질 문제 없음.
- `timings` 키 = `llm_persona`, `image_generator`, `generated_upload` — **`translate_appearance` 없음** = 단계 제거 확인.

**주의**: 이 호출은 콜드 런이라 각 단계에 모델 콜드 로드가 포함된다(워밍 아님). 'before'(번역 포함)는 코드에서 제거되어 운영 재측정이 불가하므로 §5의 구조적 비교(추정)는 그대로 두고, 여기서는 'after'의 실측만 기록한다.


In [ ]:
# 부록 A: 배포 후 라이브 실측 (콜드 런). 단위: 초(s). 출처: /v1/character 1회 응답의 timings.
measured_live_s = {
    'llm_persona (character LLM, 콜드)':   64.903,
    'image_generator (SDXL 30스텝, 콜드)':  93.124,
    'generated_upload (S3)':                1.662,
}
print('단계별(초):')
for k,v in measured_live_s.items():
    print(f'  {k:34.34} {v:7.1f}')
print(f'  {"총(콜드)":34.34} {sum(measured_live_s.values()):7.1f}')
show_bars('캐릭터 생성 단계별 라이브 실측 (콜드 런)', list(measured_live_s), list(measured_live_s.values()), unit='s')


**해석**
- 이미지 단계 콜드가 **~93초**(보고된 '5분'이 아님) → **가중치 bake가 실제로 작동 중**임을 라이브로 확인. 즉 남은 큰 레버는 콜드 제거보다 §6.3의 **워밍 추론 바닥 낮추기**(토큰·diffusion 스텝·GPU)다.
- warm 실측(콜드 제거분)은 워커가 살아있는 상태에서 재호출해 후속으로 채운다.


## 참고
- PR #121 — `perf: RunPod 콜드스타트·캐릭터 파이프라인 최적화` (mong-studio/mongle-ai)
- ADR-0004 — `docs/adr/0004-runpod-worker-build-cold-start-optimization.md`
- RunPod Serverless Health API — `GET /v2/{endpoint}/health`
- 측정 원본: 본 세션 `delayTime`/`executionTime` 실측 (2026-06-21)
